# 07 - BERTopic sobre el subcorpus de salud

In [1]:
# ============================================================
# CELL 0 - CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosPropios')

NR_TOPICS_GRID  = [10, 20, 30, 40, 60, "auto"]
NR_TOPICS_FINAL = 20
MIN_TOPIC_SIZE  = 15

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


[CONFIG] OK
  DATA_PROCESSED : C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosPropios


In [2]:
# ============================================================
# CELL 0b - COMPATIBILIDAD NUMPY 2.x / TENSORFLOW
# ============================================================
import sys, types, importlib.machinery

if 'tensorflow' not in sys.modules:
    def _fake_mod(name):
        m = types.ModuleType(name)
        m.__spec__    = importlib.machinery.ModuleSpec(name, loader=None)
        m.__path__    = []
        m.__package__ = name
        m.__version__ = '0.0.0'
        return m
    for _n in [
        'tensorflow', 'tensorflow.python', 'tensorflow.python.eager',
        'tensorflow.python.framework', 'tensorflow.python.client',
        'tensorflow.python.util', 'tensorflow.python.ops',
        'tensorflow.core', 'tensorflow.keras', 'tensorflow.keras.layers',
        'tensorflow.keras.models', 'tensorflow.keras.callbacks',
        'tensorflow.keras.losses', 'tensorflow.keras.optimizers',
        'tensorflow.compat', 'tensorflow.compat.v1', 'tensorflow.compat.v2',
    ]:
        sys.modules[_n] = _fake_mod(_n)
    print('[COMPAT] tensorflow reemplazado con modulos ficticios')
else:
    print('[COMPAT] tensorflow ya estaba cargado, sin cambios')


[COMPAT] tensorflow reemplazado con modulos ficticios


In [3]:
# ============================================================
# CELL 1 - IMPORTS Y CARGA
# ============================================================
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

corpus = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')

tweets_etiquetados = pd.read_parquet(DATA_PROCESSED / 'tweets_etiquetados_final.parquet')

tweets_salud = tweets_etiquetados[tweets_etiquetados['etiqueta_salud'] == 1].copy()

df_salud = tweets_salud.merge(
    corpus[['id_doc', 'Texto_limpio']],
    on='id_doc', how='left'
)

print(f'Corpus completo    : {len(corpus):,} tweets')
print(f'Subcorpus de salud : {len(df_salud):,} tweets')
df_salud.head(2)


Corpus completo    : 151,424 tweets
Subcorpus de salud : 29,230 tweets


,chunk_id,id_doc,texto_chunk,Salud_Control_alimentos,Salud_Economia_salud,Salud_Epidemiologia,Salud_Estadisticas_sanitarias,Salud_Higiene,Salud_Higiene_ambiental,Salud_Lucha_enfermedades,Salud_Materno_infantil,Salud_Mental,Salud_Politica_drogas,Salud_Salud_mujer,Salud_Toxicomania,score_max,subcat_max,etiqueta_salud,categoria_detectada,Texto_limpio
0,4_1,4,"Normas que deberán cumplirse en las empresas, ...",0.384811,0.318987,0.45166,0.329303,0.388348,0.346759,0.380169,0.231097,0.014135,0.340573,0.188634,0.056038,0.451660,Salud_Epidemiologia,1,Salud_Epidemiologia,"Normas que deberán cumplirse en las empresas, ..."
1,6_1,6,La otra semana ya saldrá el aumento de contagi...,0.301033,0.242711,0.42305,0.349399,0.381581,0.292549,0.477993,0.356368,0.066703,0.219171,0.243565,0.102819,0.477993,Salud_Lucha_enfermedades,1,Salud_Lucha_enfermedades,La otra semana ya saldrá el aumento de contagi...


In [4]:
# ============================================================
# CELL 2 - CARGAR Y FILTRAR EMBEDDINGS
# ============================================================
embeddings_all = np.load(DATA_PROCESSED / 'tweet_embeddings.npy')
print(f'Embeddings totales  : {embeddings_all.shape}')

mask_salud       = tweets_etiquetados['etiqueta_salud'] == 1
embeddings_salud = embeddings_all[mask_salud.values]

print(f'Embeddings de salud : {embeddings_salud.shape}')
assert len(embeddings_salud) == len(df_salud), (
    f"Mismatch: {len(embeddings_salud)} embeddings vs {len(df_salud)} tweets"
)
print('OK - embeddings alineados correctamente')


Embeddings totales  : (151424, 768)
Embeddings de salud : (29230, 768)
OK - embeddings alineados correctamente


In [5]:
# ============================================================
# CELL 3 - STOPWORDS Y VECTORIZADOR
# ============================================================
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
spanish_stopwords = list(nlp.Defaults.stop_words)

twitter_sw = [
    'rt', 'http', 'https', 'co', 'amp', 'via', 'q', 'xq', 'x',
    'si', 'ya', 'asi', 'tan', 'ser', 'hay', 'ver', 'hoy',
]
all_stopwords = list(set(spanish_stopwords + twitter_sw))

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    max_features=5000,
    min_df=5,
    ngram_range=(1, 2),
    strip_accents=None
)

documents = df_salud['Texto_limpio'].fillna('').tolist()
print(f'Documentos para BERTopic: {len(documents):,}')


Documentos para BERTopic: 29,230


In [6]:
# ============================================================
# CELL 4 - FUNCIONES DE COHERENCIA
# ============================================================

def get_topic_words(topic_model, top_n=10):
    topics_words = []
    for topic_id, word_scores in topic_model.get_topics().items():
        if topic_id == -1:
            continue
        words = [word for word, _ in word_scores[:top_n]]
        topics_words.append(words)
    return topics_words


def compute_bertopic_coherence(topic_model, documents, vectorizer_model, top_n_words=10):
    analyzer       = vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in documents]
    dictionary     = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=5, no_above=0.9)
    corpus_bow     = [dictionary.doc2bow(doc) for doc in tokenized_docs]
    topics_words   = get_topic_words(topic_model, top_n=top_n_words)
    cm = CoherenceModel(
        topics=topics_words, texts=tokenized_docs,
        dictionary=dictionary, corpus=corpus_bow, coherence='c_v'
    )
    return cm.get_coherence(), cm.get_coherence_per_topic()


In [7]:
# ============================================================
# CELL 5 - BUSQUEDA EN GRILLA DE nr_topics
# ============================================================
results = []

for nr in NR_TOPICS_GRID:
    print(f'\n--- nr_topics={nr} ---')
    tm = BERTopic(
        vectorizer_model=vectorizer_model,
        nr_topics=nr,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
        verbose=False,
        language="spanish"
    )
    topics, _ = tm.fit_transform(documents, embeddings=embeddings_salud)
    new_topics = tm.reduce_outliers(
        documents, topics, strategy="c-tf-idf", embeddings=embeddings_salud
    )
    tm.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

    coherence, _ = compute_bertopic_coherence(tm, documents, vectorizer_model)
    n_outliers   = sum(1 for t in tm.topics_ if t == -1)
    results.append({
        "nr_topics"     : nr,
        "n_topics_final": len([t for t in tm.get_topics() if t != -1]),
        "coherence_c_v" : coherence,
        "pct_outliers"  : n_outliers / len(documents)
    })
    r = results[-1]
    print(f'  Topicos: {r["n_topics_final"]}  Coh: {r["coherence_c_v"]:.4f}  Outliers: {r["pct_outliers"]:.2%}')

df_grid = pd.DataFrame(results)
print()
print(df_grid.sort_values("coherence_c_v", ascending=False).to_string(index=False))



--- nr_topics=10 ---


2026-06-02 20:07:46,693 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 9  Coh: 0.5871  Outliers: 0.01%

--- nr_topics=20 ---


2026-06-02 20:08:19,085 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 19  Coh: 0.5811  Outliers: 0.00%

--- nr_topics=30 ---


...

  Topicos: 39  Coh: 0.6079  Outliers: 0.00%

--- nr_topics=60 ---


2026-06-02 20:09:54,380 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 59  Coh: 0.6525  Outliers: 0.00%

--- nr_topics=auto ---


2026-06-02 20:10:24,641 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 61  Coh: 0.6303  Outliers: 0.00%

nr_topics  n_topics_final  coherence_c_v  pct_outliers
       60              59       0.652530      0.000000
     auto              61       0.630345      0.000000
       40              39       0.607920      0.000000
       10               9       0.587112      0.000068
       20              19       0.581081      0.000000
       30              29       0.522869      0.000000


In [8]:
# ============================================================
# CELL 6 - MODELO FINAL
# ============================================================
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics=NR_TOPICS_FINAL,
    min_topic_size=MIN_TOPIC_SIZE,
    calculate_probabilities=False,
    verbose=True,
    language="spanish"
)

topics, probs = topic_model.fit_transform(documents, embeddings=embeddings_salud)
new_topics    = topic_model.reduce_outliers(
    documents, topics, strategy="c-tf-idf", embeddings=embeddings_salud
)
topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

tm_res = topic_model.get_topic_info()
print(tm_res.head(10))


2026-06-02 20:10:40,364 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-02 20:10:45,677 - BERTopic - Dimensionality - Completed ✓
2026-06-02 20:10:45,677 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-02 20:10:46,424 - BERTopic - Cluster - Completed ✓
2026-06-02 20:10:46,426 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-02 20:10:47,835 - BERTopic - Representation - Completed ✓
2026-06-02 20:10:47,835 - BERTopic - Topic reduction - Reducing number of topics
2026-06-02 20:10:47,851 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-02 20:10:49,228 - BERTopic - Representation - Completed ✓
2026-06-02 20:10:49,232 - BERTopic - Topic reduction - Reduced number of topics from 156 to 20
2026-06-02 20:10:49,728 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure th

   Topic  Count                                             Name  \
0      0   6755  0_coronavirus_coronavirus covid19_china_covid19   
1      1   6319             1_casos_contagios_covid19_fallecidos   
2      2   3680           2_desinfección_evitar_medidas_contagio   
3      3   4351        3_pandemia_virus_pandemia covid19_covid19   
4      4   2155            4_vacuna_vacuna covid19_china_vacunas   
5      5   1695                5_salud_médicos_covid19_sanitaria   
6      6    725              6_totales_recuperados_muertes_casos   
7      7    927     7_medicamentos_tratamiento_pacientes_covid19   
8      8    355               8_mujeres_violencia_género_covid19   
9      9    399                   9_hospital_bebé_positivo_madre   

                                      Representation  \
0  [coronavirus, coronavirus covid19, china, covi...   
1  [casos, contagios, covid19, fallecidos, colomb...   
2  [desinfección, evitar, medidas, contagio, covi...   
3  [pandemia, virus, pandem

In [9]:
# ============================================================
# CELL 7 - TABLA DE RESULTADOS FINAL
# ============================================================
coh_global, coh_per_topic = compute_bertopic_coherence(
    topic_model=topic_model,
    documents=documents,
    vectorizer_model=vectorizer_model,
    top_n_words=10
)
print(f'Coherencia global (c_v): {coh_global:.4f}')

topic_ids    = [t for t in topic_model.get_topics() if t != -1]
coherence_df = pd.DataFrame({"Topic": topic_ids, "Coherence_c_v": coh_per_topic})

total_docs  = tm_res["Count"].sum()
top         = tm_res.sort_values("Count", ascending=False).head(20).copy()
top["Keywords"]   = top["Representation"].apply(lambda ws: ", ".join(ws))
top["Porcentaje"] = (top["Count"] / total_docs * 100).round(2)
top = top.merge(coherence_df, on="Topic", how="left")

final_table = top[["Topic", "Count", "Porcentaje", "Coherence_c_v", "Keywords"]]
pd.set_option("display.max_colwidth", None)
print(final_table.to_string(index=False))


Coherencia global (c_v): 0.5646
 Topic  Count  Porcentaje  Coherence_c_v                                                                                                                        Keywords
     0   6755       23.11       0.351934                    coronavirus, coronavirus covid19, china, covid19, covid19 coronavirus, casos, virus, pandemia, wuhan, madrid
     1   6319       21.62       0.532388                       casos, contagios, covid19, fallecidos, colombia, casos covid19, personas, muertes, contagiados, positivos
     3   4351       14.89       0.314291                                         pandemia, virus, pandemia covid19, covid19, covid, covid 19, 19, gripe, personas, salud
     2   3680       12.59       0.661001                     desinfección, evitar, medidas, contagio, covid19, prevención, agua, contagio covid19, propagación, prevenir
     4   2155        7.37       0.718455                           vacuna, vacuna covid19, china, vacunas, covid19, fase, v

In [10]:
# ============================================================
# CELL 8 - ASIGNAR TOPICO Y GUARDAR
# ============================================================
import pickle

df_salud = df_salud.copy()
df_salud['topico'] = topic_model.topics_

coherence_table = (
    tm_res[tm_res["Topic"] != -1].merge(coherence_df, on="Topic", how="left")
)

ruta_excel = DATA_PROCESSED / 'bertopic_salud_resultados.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    final_table.to_excel(writer,     sheet_name='Top_topicos',          index=False)
    coherence_table.to_excel(writer, sheet_name='Coherencia_por_topico',index=False)
    df_grid.to_excel(writer,         sheet_name='Grid_nr_topics',       index=False)
    df_salud[['id_doc', 'topico', 'categoria_detectada']].to_excel(
        writer, sheet_name='Tweets_por_topico', index=False
    )
print('[GUARDADO] bertopic_salud_resultados.xlsx')

df_salud.to_parquet(DATA_PROCESSED / 'salud_tweets_bertopic.parquet', index=False)
print('[GUARDADO] salud_tweets_bertopic.parquet')

print()
print('Notebook 07 completado.')
print('Siguiente -> 08_extraer_frecuencias_POS.ipynb')


[GUARDADO] bertopic_salud_resultados.xlsx
[GUARDADO] salud_tweets_bertopic.parquet

Notebook 07 completado.
Siguiente -> 08_extraer_frecuencias_POS.ipynb
